# TP3 - Prédiction de la consommation de carburant avec LSTM / TinyML
**Rapport associé :** `rapport_LSTM.pdf` - ES-SAOUDY Zakaria

Reproduction stricte : 638 véhicules valides, 3 variables, split 70/30, deux LSTM de 12 unités,
Adamax, MAE, batch 1, 300 epochs, diagnostics des résidus et export `eloquent_tensorflow`.

## 1. Introduction et dataset
Le modèle prédit `FUEL CONSUMPTION` à partir de `ENGINE SIZE`, `CYLINDERS` et `COEMISSIONS`.
Le dataset Kaggle contient 639 véhicules de l'année 2000 et 10 variables ; un doublon est retiré.
La finalité annoncée est un déploiement TinyML sur ARM Cortex-M.

In [ ]:
%pip install -q tensorflow pandas scikit-learn seaborn eloquent-tensorflow==1.0.8

In [ ]:
import urllib.request, zipfile
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
tf.keras.utils.set_random_seed(42)

## 2. Téléchargement, inspection et nettoyage

In [ ]:
DATA = Path('data'); DATA.mkdir(exist_ok=True)
csv_path = DATA / 'FuelConsumption.csv'
if not csv_path.exists():
    archive = DATA / 'fuelconsumption.zip'
    urllib.request.urlretrieve(
        'https://www.kaggle.com/api/v1/datasets/download/krupadharamshi/fuelconsumption', archive)
    with zipfile.ZipFile(archive) as bundle: bundle.extractall(DATA)
    downloaded = list(DATA.glob('*.csv'))[0]; downloaded.rename(csv_path)
df = pd.read_csv(csv_path)
print(df.head()); print(df.info()); print(df.describe())
before = len(df); df.dropna(inplace=True); df.drop_duplicates(inplace=True)
print(f'Nettoyage : {before} -> {len(df)} lignes'); assert len(df) == 638

## 3. Analyse exploratoire
Le pairplot examine les relations bivariées. La corrélation de Spearman, robuste aux outliers, met en
évidence une corrélation d'environ 0.97 entre consommation et émissions de CO2.

In [ ]:
eda_cols = ['ENGINE SIZE','CYLINDERS','FUEL CONSUMPTION','COEMISSIONS ']
sns.pairplot(df[eda_cols]); plt.show()
corr = df[eda_cols].corr('spearman')
plt.figure(figsize=(18,10)); heatmap = sns.heatmap(corr, xticklabels=corr.columns,
    yticklabels=corr.columns, cmap='coolwarm')
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j+.5, i+.5, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=18)
plt.show(); print(corr)

## 4. Split 70/30 et reshape `(samples, 3, 1)` - sans normalisation, comme le rapport

In [ ]:
X = df[['ENGINE SIZE','CYLINDERS','COEMISSIONS ']]
y = df[['FUEL CONSUMPTION']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state=42)
X_train = np.array(X_train).reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = np.array(X_test).reshape(X_test.shape[0], X_test.shape[1], 1)
y_train_np = y_train.to_numpy(); y_test_np = y_test.to_numpy()
print(X_train.shape, X_test.shape); assert X_train.shape == (446,3,1) and X_test.shape == (192,3,1)

## 5. Architecture
Input `(3,1)` -> LSTM 12 avec séquence -> LSTM 12 -> Dense 32 ReLU -> Dense 1 linéaire.
Adamax est choisi pour sa stabilité sur petit dataset ; MAE pour sa robustesse aux outliers.

In [ ]:
def instantiate_lstm_for_regression(input_shape, output_shape):
    model = tf.keras.Sequential()
    model.add(layers.Input(shape=input_shape, batch_size=1))
    model.add(layers.LSTM(12, unroll=False, return_sequences=True))
    model.add(layers.LSTM(12, unroll=False))
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dense(output_shape, activation='linear'))
    model.compile(optimizer='adamax', loss='mae', metrics=['mae'])
    return model
model = instantiate_lstm_for_regression((3,1), 1)
model.summary(); assert model.count_params() == 2321

## 6. Entraînement exact - validation 20 %, 300 epochs, batch 1

In [ ]:
history = model.fit(X_train, y_train_np, validation_split=.2, epochs=300, batch_size=1, verbose=2)
plt.plot(history.history['loss'], 'r.', label='Training loss')
plt.plot(history.history['val_loss'], color='#d4a900', label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('MAE'); plt.legend(); plt.grid(); plt.show()

## 7. Évaluation, valeurs réelles/prédites et résidus

In [ ]:
test_loss, test_mae = model.evaluate(X_test, y_test_np, batch_size=1, verbose=0)
y_train_pred = model.predict(X_train, batch_size=1, verbose=0)
y_test_pred = model.predict(X_test, batch_size=1, verbose=0)
print(f'MAE train finale : {history.history["loss"][-1]:.4f}')
print(f'MAE test : {test_mae:.4f}')

fig, axes = plt.subplots(1,2,figsize=(14,4))
axes[0].plot(y_train_np, label='original'); axes[0].plot(y_train_pred, label='predicted'); axes[0].set_title('Train')
axes[1].plot(y_test_np, label='original'); axes[1].plot(y_test_pred, label='predicted'); axes[1].set_title('Test')
for ax in axes: ax.legend(); ax.grid()
plt.show()

train_residuals = y_train_np.reshape(-1) - y_train_pred.reshape(-1)
test_residuals = y_test_np.reshape(-1) - y_test_pred.reshape(-1)
fig, axes = plt.subplots(2,2,figsize=(14,10))
for ax, pred, res, color, title in [
    (axes[0,0], y_train_pred, train_residuals, 'blue', 'Train résidus'),
    (axes[0,1], y_test_pred, test_residuals, 'green', 'Test résidus')]:
    mean, std = np.mean(res), np.std(res); ax.scatter(pred, res, c=color, marker='o')
    ax.axhline(0,color='r'); ax.axhline(mean,color='k',linestyle='--')
    ax.axhline(mean+2*std,color='g',linestyle='--'); ax.axhline(mean-2*std,color='g',linestyle='--'); ax.set_title(title)
axes[1,0].hist(train_residuals,bins=20,color='blue',alpha=.6); axes[1,0].set_title('Histogramme train')
axes[1,1].hist(test_residuals,bins=20,color='green',alpha=.6); axes[1,1].set_title('Histogramme test')
for ax, res in [(axes[1,0], train_residuals), (axes[1,1], test_residuals)]:
    mean, std = np.mean(res), np.std(res)
    ax.axvline(mean,color='k',linestyle='--'); ax.axvline(mean+2*std,color='g',linestyle='--')
    ax.axvline(mean-2*std,color='g',linestyle='--')
plt.tight_layout(); plt.show()

## 8. Export microcontrôleur - code du rapport

In [ ]:
from eloquent_tensorflow import convert_model
code = convert_model(model)
Path('LSTM').mkdir(exist_ok=True)
with open('./LSTM/model.h', 'w') as file: file.write(code)
print('Header généré : ./LSTM/model.h | caractères :', len(code))

## 9. Discussion, limites et perspectives
Résultats rapportés : MAE train `1.1045`, MAE test `1.0334`, moyenne des résidus proche de zéro et
écart-type d'environ `1.05` (train) / `1.02` (test). Points forts : nettoyage, généralisation,
résidus centrés et architecture compacte.
Avantages TinyML : faible latence, fonctionnement hors-ligne, confidentialité et faible consommation.

Limites : LSTM suboptimal pour données tabulaires i.i.d., véhicules de 2000 seulement, absence de
normalisation et absence de benchmark. Améliorations : `StandardScaler`/`MinMaxScaler`, données
multi-années et comparaison régression linéaire / Random Forest / MLP / LSTM, avec coût embarqué mesuré.